# Initial Libraies

In [1]:
!pip install dwave-ocean-sdk

In [2]:
!pip install dwave-neal

In [3]:
!pip install job-shop-lib

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from job_shop_lib.visualization.gantt import plot_gantt_chart
from job_shop_lib import JobShopInstance, Operation
from job_shop_lib.dispatching import (
    ReadyOperationsFilterType,
    create_composite_operation_filter,
)
from job_shop_lib.dispatching.rules import (
    DispatchingRuleSolver,
    DispatchingRuleType,
)

from job_shop_lib.benchmarking import load_benchmark_instance

from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm

from collections import Counter

from dwave.system.samplers import DWaveSampler
from dwave.system.composites import EmbeddingComposite
import dwave.inspector as inspector
from dimod import ConstrainedQuadraticModel, CQM, SampleSet
from dwave.system import LeapHybridCQMSampler
from dimod.vartypes import Vartype
from dimod import Binary, quicksum
from dimod import BinaryQuadraticModel
import dimod

from neal import SimulatedAnnealingSampler

In [5]:
plt.style.use("ggplot")

In [6]:
def plot_gantt_chart_from_dispatching_rule(
    dispatching_rule: DispatchingRuleType,
    instance: JobShopInstance,
    optimization_level: int = 0,
) -> plt.Figure:

    if optimization_level == 0:
        pruning_function = None
    elif optimization_level == 1:
        pruning_function = ReadyOperationsFilterType.DOMINATED_OPERATIONS
    elif optimization_level == 2:
        pruning_function = create_composite_operation_filter(
            [
                ReadyOperationsFilterType.DOMINATED_OPERATIONS,
                ReadyOperationsFilterType.NON_IMMEDIATE_MACHINES,
            ]
        )
    else:
        raise ValueError(f"Invalid optimization level: {optimization_level}")

    solver = DispatchingRuleSolver(
        dispatching_rule,
        ready_operations_filter=pruning_function,
    )
    solution = solver.solve(instance)

    title = f"{instance.name} - {dispatching_rule.name} (optimization level {optimization_level})"
    number_of_x_ticks = 10 if solution.makespan() >= 1000 else 15
    fig, ax = plot_gantt_chart(
        solution, title=title, number_of_x_ticks=number_of_x_ticks
    )
    if instance.num_jobs > 20:
        # Remove legend if there are too many jobs
        ax.legend().remove()
    return fig

In [7]:
class JSPMUInstance:

    def __init__(self, name=""):
        self.name = name
        self.n_jobs = 0
        self.n_machines = 0

        # jobs[j] = [(machine, duration), ...]
        self.jobs = []

        # machine_holes[m] = [(start, end), ...]
        self.machine_holes = {}

    @staticmethod
    def read(filepath):

        instance = JSPMUInstance(Path(filepath).stem)

        with open(filepath, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # ---- header ----
        header = lines[0].split()
        instance.n_jobs = int(header[0])
        instance.n_machines = int(header[1])

        # ---- jobs ----
        for j in range(instance.n_jobs):

            tokens = list(map(int, lines[1 + j].split()))

            if len(tokens) != 2 * instance.n_machines:
                raise ValueError(
                    f"Job {j} has wrong number of entries."
                )

            ops = []

            for k in range(0, len(tokens), 2):

                machine = tokens[k]
                duration = tokens[k + 1]

                if machine < 0 or machine >= instance.n_machines:
                    raise ValueError(f"Invalid machine {machine}")

                ops.append((machine, duration))

            instance.jobs.append(ops)

        # ---- initialize empty holes ----
        for m in range(instance.n_machines):
            instance.machine_holes[m] = []

        # ---- holes section ----
        idx = 1 + instance.n_jobs

        if idx < len(lines):

            if lines[idx] != "[MACHINE_HOLES]":
                raise ValueError("Missing [MACHINE_HOLES] section")

            idx += 1

            while idx < len(lines):

                tokens = list(map(int, lines[idx].split()))

                machine = tokens[0]
                n_holes = tokens[1]

                expected = 2 + 2 * n_holes
                if len(tokens) != expected:
                    raise ValueError(
                        f"Wrong hole format for machine {machine}"
                    )

                holes = []

                pos = 2
                for _ in range(n_holes):

                    start = tokens[pos]
                    duration = tokens[pos + 1]

                    if duration <= 0:
                        raise ValueError("Hole duration must be positive")

                    holes.append((start, start + duration))

                    pos += 2

                instance.machine_holes[machine] = holes

                idx += 1

        return instance

    def __str__(self) -> str:
        print('Name:',self.name)
        print("n jobs:",self.n_jobs)
        print("n machines:",self.n_machines)
        print("jobs:")
        for j in self.jobs:
          print(j)
        print("Unavailabilities:")
        for h in self.machine_holes:
          print('Machine',h,':',self.machine_holes[h])
        return ''

    def get_naive_t(self):
      naive_t = 0
      for j in self.jobs:
        durs = [op[1] for op in j]
        naive_t += sum(durs)
      return naive_t

In [8]:
def to_job_shop_lib_instance(jspmu_instance, name=None, accept_unavailabilities = False):
    """
    Convert a JSPMUInstance into a job_shop_lib.JobShopInstance.

    Conventions:
    - Original jobs keep their original routing.
    - Each machine unavailability (start, end) becomes a dummy one-operation job.
    - metadata["release_dates"]:
        * 0 for original jobs
        * hole start for dummy maintenance jobs
    - metadata["deadlines"]:
        * float("inf") for original jobs
        * hole end for dummy maintenance jobs
    """

    jobs = []
    release_dates = []
    deadlines = []

    # --- Original jobs ---
    # jspmu_instance.jobs[j] = [(machine, duration), ...]
    for job in jspmu_instance.jobs:
        converted_job = [Operation(machine, duration) for machine, duration in job]
        jobs.append(converted_job)
        release_dates.append(0)
        deadlines.append(float("inf"))

    if accept_unavailabilities:
      # --- Dummy jobs for machine holes ---
      # jspmu_instance.machine_holes[m] = [(start, end), ...]
      # Each hole becomes a 1-operation job of duration end-start on machine m
      for machine in sorted(jspmu_instance.machine_holes):
          holes = jspmu_instance.machine_holes[machine]
          for start, end in holes:
              duration = end - start
              if duration <= 0:
                  raise ValueError(
                      f"Invalid hole on machine {machine}: ({start}, {end})"
                  )

              dummy_job = [Operation(machine, duration)]
              jobs.append(dummy_job)
              release_dates.append(start)
              deadlines.append(end)

    instance = JobShopInstance(
        jobs=jobs,
        name=name if name is not None else jspmu_instance.name
    )

    instance.metadata["release_dates"] = release_dates
    instance.metadata["deadlines"] = deadlines

    return instance

# Create BQM

In [9]:
def build_jspmu_bqm(
    inst,
    horizon,
    lambda_1=10.0,
    lambda_2=10.0,
    lambda_3=10.0,
):
    """
    Build a BINARY BQM for a Job Shop Scheduling Problem with machine unavailabilities,
    following the paper's approach:
      - real operations are modeled with time-indexed binary variables x[j,i,t]
      - fixed machine unavailabilities are treated as fixed dummy one-operation jobs
        and are incorporated through the machine-conflict penalty P2

    Parameters
    ----------
    inst : object
        Instance object with attributes:
          - inst.jobs : list of jobs; each job is a list of (machine, proc_time)
          - inst.machine_holes : dict-like or list-like structure such that
                                 inst.machine_holes[m] = [(start, end), ...]
          - inst.n_jobs
          - inst.n_machines
    horizon : int
        Time horizon H. Start times are allowed in {0, ..., H-p}.
    lambda_1 : float
        Penalty weight for "start exactly once" (P1).
    lambda_2 : float
        Penalty weight for machine conflicts (P2), including conflicts with
        fixed machine unavailability intervals.
    lambda_3 : float
        Penalty weight for within-job precedence (P3).

    Returns
    -------
    bqm : dimod.BinaryQuadraticModel
        BQM in vartype BINARY.
    metadata : dict
        Helpful metadata about variables and operations.
    """

    if horizon <= 0:
        raise ValueError("horizon must be a positive integer")

    jobs = inst.jobs
    n_jobs = len(jobs)

    # --- helper functions -----------------------------------------------------

    def var_label(j, i, t):
        # j = job index, i = operation index within job, t = start time
        return (j, i, t)

    def intervals_overlap(s1, p1, s2, p2):
        """Return True if [s1, s1+p1) overlaps [s2, s2+p2)."""
        return (s1 < s2 + p2) and (s2 < s1 + p1)

    def get_holes(machine):
        # works whether machine_holes is a dict or a list
        holes = inst.machine_holes[machine]
        return holes if holes is not None else []

    # --- collect operation data -----------------------------------------------

    operations = []  # each element: dict with job, op, machine, p
    machine_to_ops = defaultdict(list)

    for j, job in enumerate(jobs):
        for i, (machine, p) in enumerate(job):
            if p <= 0:
                raise ValueError(f"Processing time must be positive, got p={p} for job {j}, op {i}")
            if p > horizon:
                raise ValueError(
                    f"Operation (job={j}, op={i}) has p={p} > horizon={horizon}; "
                    "increase horizon."
                )

            op = {
                "job": j,
                "op": i,
                "machine": machine,
                "p": p,
                "valid_starts": list(range(0, horizon - p + 1)),
            }
            operations.append(op)
            machine_to_ops[machine].append(op)

    # --- create BQM -----------------------------------------------------------

    bqm = dimod.BinaryQuadraticModel(vartype=dimod.BINARY)

    # Add all variables explicitly with 0 linear bias
    for op in operations:
        j, i = op["job"], op["op"]
        for t in op["valid_starts"]:
            bqm.add_variable(var_label(j, i, t), 0.0)

    # -------------------------------------------------------------------------
    # Objective: minimize sum of start times of the last operation of each job
    # f(x) = sum_j sum_t t * x[last_op_of_job_j, t]
    # -------------------------------------------------------------------------
    for j, job in enumerate(jobs):
        last_i = len(job) - 1
        p_last = job[last_i][1]
        for t in range(0, horizon - p_last + 1):
            bqm.add_linear(var_label(j, last_i, t), float(t))

    # -------------------------------------------------------------------------
    # P1: each operation starts exactly once
    #   (sum_t x_t - 1)^2
    # For binary x:
    #   = 1 - sum_t x_t + 2 * sum_{t<t'} x_t x_t'
    # -------------------------------------------------------------------------
    for op in operations:
        j, i = op["job"], op["op"]
        starts = op["valid_starts"]

        bqm.offset += lambda_1

        for t in starts:
            bqm.add_linear(var_label(j, i, t), -lambda_1)

        for idx1 in range(len(starts)):
            t1 = starts[idx1]
            v1 = var_label(j, i, t1)
            for idx2 in range(idx1 + 1, len(starts)):
                t2 = starts[idx2]
                v2 = var_label(j, i, t2)
                bqm.add_quadratic(v1, v2, 2.0 * lambda_1)

    # -------------------------------------------------------------------------
    # P2: machine conflicts between real operations
    # Add lambda_2 * x[j,i,t] * x[j',i',t'] whenever two operations on the
    # same machine overlap in time.
    # -------------------------------------------------------------------------
    for machine, ops_on_machine in machine_to_ops.items():
        n_ops_m = len(ops_on_machine)
        for a in range(n_ops_m):
            op_a = ops_on_machine[a]
            j1, i1, p1 = op_a["job"], op_a["op"], op_a["p"]

            for b in range(a + 1, n_ops_m):
                op_b = ops_on_machine[b]
                j2, i2, p2 = op_b["job"], op_b["op"], op_b["p"]

                for t1 in op_a["valid_starts"]:
                    v1 = var_label(j1, i1, t1)
                    for t2 in op_b["valid_starts"]:
                        if intervals_overlap(t1, p1, t2, p2):
                            v2 = var_label(j2, i2, t2)
                            bqm.add_quadratic(v1, v2, lambda_2)

    # -------------------------------------------------------------------------
    # P2 extension for fixed machine unavailabilities:
    # each hole [a,b) is treated like a fixed dummy one-operation job with
    # start fixed at a and duration (b-a). Since its binary variable is fixed
    # to 1, the quadratic conflict term reduces to a linear penalty on any
    # real operation start that overlaps the hole.
    # -------------------------------------------------------------------------
    for machine, ops_on_machine in machine_to_ops.items():
        holes = get_holes(machine)

        for (a, b) in holes:
            hole_start = int(a)
            hole_end = int(b)
            hole_len = hole_end - hole_start

            if hole_len <= 0:
                raise ValueError(f"Invalid hole on machine {machine}: {(a, b)}")

            for op in ops_on_machine:
                j, i, p = op["job"], op["op"], op["p"]

                for t in op["valid_starts"]:
                    if intervals_overlap(t, p, hole_start, hole_len):
                        # conflict with fixed dummy operation => linear term
                        bqm.add_linear(var_label(j, i, t), lambda_2)

    # -------------------------------------------------------------------------
    # P3: precedence inside each job
    # Add lambda_3 * x[j,i,t] * x[j,i+1,t'] whenever t + p_i > t'
    # -------------------------------------------------------------------------
    for j, job in enumerate(jobs):
        for i in range(len(job) - 1):
            p_i = job[i][1]
            p_next = job[i + 1][1]

            starts_i = range(0, horizon - p_i + 1)
            starts_next = range(0, horizon - p_next + 1)

            for t in starts_i:
                v1 = var_label(j, i, t)
                cutoff = t + p_i  # next op cannot start before this

                for t_next in starts_next:
                    if t_next < cutoff:
                        v2 = var_label(j, i + 1, t_next)
                        bqm.add_quadratic(v1, v2, lambda_3)

    metadata = {
        "name": getattr(inst, "name", None),
        "n_jobs": n_jobs,
        "n_machines": getattr(inst, "n_machines", None),
        "num_real_operations": len(operations),
        "num_variables": bqm.num_variables,
        "num_interactions": bqm.num_interactions,
        "horizon": horizon,
        "lambdas": {
            "lambda_1": lambda_1,
            "lambda_2": lambda_2,
            "lambda_3": lambda_3,
        },
        "variable_format": "(job_index, op_index, start_time)",
    }

    return bqm, metadata

# Embedding function

In [10]:
import minorminer
import dwave_networkx as dnx

In [11]:
def embed_bqm(bqm, topology: str = 'pegasus',m_multiplier = 1.0, random_seed: int = None, time_out = None, verbose = None):
    """
    Embeds a BQM into a quantum annealer topology using the minorminer heuristic.

    Parameters
    ----------
    bqm         : dimod BQM object to embed
    topology    : 'pegasus' or 'zephyr' (case-insensitive)
    random_seed : optional random seed for reproducibility

    Returns
    -------
    embedding   : dict mapping BQM variables to lists of target qubits
    G_target    : the target hardware graph used for embedding
    m           : the topology size parameter chosen
    """

    topology = topology.lower()
    if topology not in ('pegasus', 'zephyr'):
        raise ValueError(f"Unsupported topology '{topology}'. Choose 'pegasus' or 'zephyr'.")

    if verbose is not None:
      print('finding number of topology clusters')

    num_variables = len(bqm.variables)

    # --- Determine m based on topology and BQM size ---
    if topology == 'pegasus':
        # Pegasus(m) has 24m(m-1) qubits: m=3->144, m=4->288, m=5->480...
        # Find the smallest m such that the graph has enough qubits
        for m in range(3, 1000):  # m=16 gives Pegasus P16, the largest real device
            if 24 * m * (m - 1) >= num_variables:
                break
        m = int(m*m_multiplier)
        G_target = dnx.pegasus_graph(m)

    elif topology == 'zephyr':
        # Zephyr(m) has 4m(2m+1) qubits: m=2->40, m=3->84, m=4->144, m=6->300...
        for m in range(2, 1000):  # m=8 is a generous upper bound for Zephyr
            if 4 * m * (2 * m + 1) >= num_variables:
                break
        m = int(m*m_multiplier)
        G_target = dnx.zephyr_graph(m)

    if verbose is not None:
      print(f'Number of clusters =  {m}')
      print('Transforming BQM into networkx graph')
    # --- Convert BQM to a NetworkX graph for minorminer ---
    G_problem = bqm.to_networkx_graph()

    # --- Run minorminer ---
    kwargs = {}
    if random_seed is not None:
        kwargs['random_seed'] = random_seed
    if time_out is not None:
        kwargs['timeout'] = time_out
    if verbose is not None:
        kwargs['verbose'] = verbose

    if verbose is not None:
      print('Running minor-miner...')

    embedding = minorminer.find_embedding(G_problem, G_target, **kwargs)

    if not embedding:
        raise RuntimeError(
            f"minorminer failed to find an embedding for {num_variables} variables "
            f"into {topology.capitalize()}(m={m}). "
            f"Consider increasing m manually or simplifying the BQM."
        )
    if verbose is not None:
      print(f"[embed_bqm] Topology : {topology.capitalize()}(m={m})")
      print(f"[embed_bqm] Variables: {num_variables}")
      print(f"[embed_bqm] Qubits available: {G_target.number_of_nodes()}")
      print(f"[embed_bqm] Chains used: {len(embedding)}")

    return embedding, G_target, m

In [12]:
def get_avg_chain_length(embedding):
    avg_chain_length = []
    for chain in embedding.values():
        avg_chain_length.append(len(chain))
    avg = sum(avg_chain_length)/len(avg_chain_length)
    min_chain = min(avg_chain_length)
    max_chain = max(avg_chain_length)
    std = np.std(avg_chain_length)
    total_qbits = sum(avg_chain_length)
    return total_qbits, avg, std, min_chain, max_chain

# test

In [27]:
# inst = JSPMUInstance.read("abz5_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu")

In [28]:
# print(inst)

## naive

In [29]:
# inst.get_naive_t()

In [30]:
# bqm, meta = build_jspmu_bqm(
#     inst,
#     horizon = inst.get_naive_t(),
#     lambda_1=20.0,
#     lambda_2=20.0,
#     lambda_3=20.0,
# )

In [31]:
# %%time
# embedding, G_target, m = embed_bqm(bqm,
#                                    topology = 'pegasus',
#                                    random_seed = 123,
#                                    m_multiplier = 5.0,
#                                    time_out=120,
#                                    verbose=2)

In [32]:
# get_avg_chain_length(embedding)

In [33]:
# %%time
# embedding, G_target, m = embed_bqm(bqm,
#                                    topology = 'zephyr',
#                                    random_seed = 123,
#                                    m_multiplier = 5.0,
#                                    time_out=120,
#                                    verbose=2)

In [34]:
# get_avg_chain_length(embedding)


## with H

In [35]:
# bqm, meta = build_jspmu_bqm(
#     inst,
#     horizon = 14,
#     lambda_1=20.0,
#     lambda_2=20.0,
#     lambda_3=20.0,
# )

In [36]:
# %%time
# embedding, G_target, m = embed_bqm(bqm,
#                                    topology = 'pegasus',
#                                    random_seed = 123,
#                                    m_multiplier = 5.0,
#                                    time_out=120,
#                                    verbose=2)

In [37]:
# get_avg_chain_length(embedding)

In [38]:
# %%time
# embedding, G_target, m = embed_bqm(bqm,
#                                    topology = 'zephyr',
#                                    random_seed = 123,
#                                    m_multiplier = 5.0,
#                                    time_out=120,
#                                    verbose=2)

In [39]:
# get_avg_chain_length(embedding)

In [40]:
# len(G_target.nodes)

# Embedding analysis all instances

In [41]:
import zipfile
import os
import time

In [42]:
df = pd.read_csv('heuristic_H_Non_Preemptive_results_rescaled.csv')
df.head()

,id,H-Preemptive
0,abz5,14
1,abz6,11
2,ft06,-1
3,ft10,14
4,la01,11


In [43]:
df.index = df['id']

In [44]:
makespans_H = df.rename(columns = {'H-Preemptive':'T'}).drop(columns = ['id']).T.to_dict()

In [45]:
makespans_H

{'abz5': {'T': 14},
 'abz6': {'T': 11},
 'ft06': {'T': -1},
 'ft10': {'T': 14},
 'la01': {'T': 11},
 'la02': {'T': 9},
 'la03': {'T': 11},
 'la04': {'T': 13},
 'la05': {'T': 12},
 'la16': {'T': 13},
 'la17': {'T': 10},
 'la18': {'T': 14},
 'la19': {'T': 15},
 'la20': {'T': 15},
 'orb01': {'T': 11},
 'orb02': {'T': 14},
 'orb03': {'T': 12},
 'orb04': {'T': 14},
 'orb05': {'T': 12},
 'orb06': {'T': 16},
 'orb07': {'T': 13},
 'orb08': {'T': 14},
 'orb09': {'T': 13},
 'orb10': {'T': 10}}

In [46]:
zip_path = "generated_jspmu_instances_literature_rescaled_3.zip"
extract_dir = "generated_jspmu_instances_literature_rescaled"

# List to store file names inside the zip
file_names = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_names = zip_ref.namelist()  # names of all files/folders inside the zip
    zip_ref.extractall(extract_dir)  # unzip everything

print("Files inside zip:")
print(file_names)

Files inside zip:
['abz5_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'abz6_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'ft06_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'ft10_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la01_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la02_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la03_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la04_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la05_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la16_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la17_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la18_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la19_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la20_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb01_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb02_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb03_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb04_mu_mtbf300_durlognorm35_03_op1-3_hol

In [49]:
data = []
problematics = []
embeddings_out = []

for instance_name in tqdm(file_names):
  id_instance = instance_name.split('_')[0]

  try:
    jspmu_instance = JSPMUInstance.read(f'{extract_dir}/{instance_name}')
    machine_unavailability = jspmu_instance.machine_holes
  except:
    problematics.append(instance_name)
    row = {
        'id':id_instance,
        't':-1,
        'topology':-1,
        'G_size':-1,
        'm':-1,
        'n':-1,
        'avg_chain':-1,
        'std_chain':-1,
        'min_chain':-1,
        'max_chain':-1,
        'time':-1
    }
    continue

  t_naive = jspmu_instance.get_naive_t()
  H_T = makespans_H[id_instance]['T']

  for t,tipo in zip([t_naive,H_T],['naive','H']):

    bqm, meta = build_jspmu_bqm(
    jspmu_instance,
    horizon = t,
    lambda_1=20.0,
    lambda_2=20.0,
    lambda_3=20.0,
    )

    for topology in ['pegasus', 'zephyr']:
      try:
        tic = time.time()
        embedding, G_target, m = embed_bqm(bqm,
                                    topology = topology,
                                    random_seed = 123,
                                    m_multiplier = 5.0,
                                    time_out=120,
                                    verbose=2)
        toc = time.time()
        e = toc - tic
        total_qbits, avg, std, min_chain, max_chain = get_avg_chain_length(embedding)
        row = {
            'id':id_instance,
            'horizon_type':tipo,
            't':t,
            'topology':topology,
            'G_size':len(G_target.nodes),
            'm':m,
            'n':total_qbits,
            'avg_chain':avg,
            'std_chain':std,
            'min_chain':min_chain,
            'max_chain':max_chain,
            'time':e
        }
        data.append(row)
        embeddings_out.append(
            (id_instance,topology,G_target,m,embedding)
        )
      except:
        row = {
            'id':id_instance,
            't':t,
            'topology':topology,
            'G_size':-1,
            'm':-1,
            'n':-1,
            'avg_chain':-1,
            'std_chain':-1,
            'min_chain':-1,
            'max_chain':-1,
            'time':-1
        }
        data.append(row)

  0%|          | 0/24 [00:00<?, ?it/s]

finding number of topology clusters
Number of clusters =  20
Transforming BQM into networkx graph
Running minor-miner...


/tmp/ipykernel_37172/259036506.py:49: DeprecationWarning: BinaryQuadraticModel.to_networkx_graph() is deprecated since dimod 0.10.0 and will be removed in 0.12.0. Use dimod.to_networkx_graph() instead.
  G_problem = bqm.to_networkx_graph()


[embed_bqm] Topology : Pegasus(m=20)
[embed_bqm] Variables: 169
[embed_bqm] Qubits available: 8968
[embed_bqm] Chains used: 169
finding number of topology clusters
Number of clusters =  25
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Zephyr(m=25)
[embed_bqm] Variables: 169
[embed_bqm] Qubits available: 20400
[embed_bqm] Chains used: 169
finding number of topology clusters
Number of clusters =  15
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Pegasus(m=15)
[embed_bqm] Variables: 115
[embed_bqm] Qubits available: 4928
[embed_bqm] Chains used: 115
finding number of topology clusters
Number of clusters =  20
Transforming BQM into networkx graph
Running minor-miner...


  4%|▍         | 1/24 [06:31<2:29:53, 391.01s/it]

...

[embed_bqm] Topology : Zephyr(m=20)
[embed_bqm] Variables: 116
[embed_bqm] Qubits available: 13120
[embed_bqm] Chains used: 116
finding number of topology clusters
Number of clusters =  20
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Pegasus(m=20)
[embed_bqm] Variables: 169
[embed_bqm] Qubits available: 8968
[embed_bqm] Chains used: 169
finding number of topology clusters
Number of clusters =  25
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Zephyr(m=25)
[embed_bqm] Variables: 169
[embed_bqm] Qubits available: 20400
[embed_bqm] Chains used: 169
finding number of topology clusters
Number of clusters =  15
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Pegasus(m=15)
[embed_bqm] Variables: 106
[embed_bqm] Qubits available: 4928
[embed_bqm] Chains used: 106
finding number of topology clusters
Number of clusters =  20
Transforming BQM into networkx graph
Running minor-miner...


 96%|█████████▌| 23/24 [1:43:12<05:18, 318.76s/it]

[embed_bqm] Topology : Zephyr(m=20)
[embed_bqm] Variables: 106
[embed_bqm] Qubits available: 13120
[embed_bqm] Chains used: 106
finding number of topology clusters
Number of clusters =  20
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Pegasus(m=20)
[embed_bqm] Variables: 145
[embed_bqm] Qubits available: 8968
[embed_bqm] Chains used: 145
finding number of topology clusters
Number of clusters =  25
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Zephyr(m=25)
[embed_bqm] Variables: 145
[embed_bqm] Qubits available: 20400
[embed_bqm] Chains used: 145
finding number of topology clusters
Number of clusters =  15
Transforming BQM into networkx graph
Running minor-miner...
[embed_bqm] Topology : Pegasus(m=15)
[embed_bqm] Variables: 82
[embed_bqm] Qubits available: 4928
[embed_bqm] Chains used: 82
finding number of topology clusters
Number of clusters =  15
Transforming BQM into networkx graph
Running minor-miner...


100%|██████████| 24/24 [1:47:49<00:00, 269.58s/it]

[embed_bqm] Topology : Zephyr(m=15)
[embed_bqm] Variables: 82
[embed_bqm] Qubits available: 7440
[embed_bqm] Chains used: 82


In [52]:
out_df = pd.DataFrame(data)

In [53]:
out_df.to_csv('chain_and_embedding_results.csv')

# Save embeddings to pickle

In [54]:
import pickle

In [55]:
with open('embeddings.pickle', 'wb') as handle:
    pickle.dump(embeddings_out, handle, protocol=pickle.HIGHEST_PROTOCOL)
